# Five experiments that go beyond running the example

Prepared Python workshop for experienced analysts. All cases are synthetic, independent of the original notebook datasets. No downloads or accounts. Run the setup, then the relevant day's section. Change one assumption at a time; record your prediction before running and explain what changed afterwards.

These experiments replace part of Lab C; they do not add hours to the timetable. Keep model selection separate from final evaluation. The supplied thresholds and costs are teaching choices, not standards.


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import beta
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split


## Day 1 - Is the ranking robust?

Two service teams have different mixes of routine and complex work. Predict which team looks faster overall and which is faster within each type. Compute the pooled mean from counts and means. Then compare both teams under the same 50/50 mix. Explain why a reversal can occur without any arithmetic error.

**Your experiment:** change the common complex-case share to 0.2 and 0.8. State what remains stable. Do not claim the comparison proves a causal effect of assigning work to a team.


In [ ]:
mix = pd.DataFrame({
    'team': ['A', 'A', 'B', 'B'],
    'type': ['routine', 'complex', 'routine', 'complex'],
    'count': [90, 10, 10, 90],
    'mean_minutes': [10., 30., 8., 25.]
})
mix['total_minutes'] = mix['count'] * mix['mean_minutes']
pooled = mix.groupby('team')[['count', 'total_minutes']].sum()
pooled['pooled_mean'] = pooled['total_minutes'] / pooled['count']
common_complex_share = 0.5
by_type = mix.pivot(index='team', columns='type', values='mean_minutes')
standardised = ((1-common_complex_share)*by_type['routine']
                + common_complex_share*by_type['complex'])
print(mix.to_string(index=False))
print('Observed mix means:', pooled['pooled_mean'].round(2).to_dict())
print('Common mix means:', standardised.round(2).to_dict())


## Day 2 - Choose an operating threshold, not just a model

This separate experiment has train, validation and final test partitions. Select an alert threshold using validation error costs. The candidate grid is fixed before looking at the test. The final block evaluates the selected threshold once.

**Your experiment:** on validation data only, compare missed-event costs of 10, 50 and 100 with a false-alarm cost of 5. Would one threshold suit all three policies? What costs have we left out? Once you inspect the final test, further changes need a fresh final evaluation; rerunning that same test is not new evidence.


In [ ]:
rng = np.random.default_rng(23)
X = rng.normal(size=(1200, 3))
prob = 1/(1+np.exp(-(-1.1 + 1.4*X[:, 0] - 0.8*X[:, 1])))
y = rng.binomial(1, prob)
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=.2, random_state=31, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=.25, random_state=32, stratify=y_dev)
model = LogisticRegression().fit(X_train, y_train)
p_val = model.predict_proba(X_val)[:, 1]
thresholds = np.arange(.1, .91, .1)
def errors(actual, predicted):
    tn, fp, fn, tp = confusion_matrix(actual, predicted, labels=[0,1]).ravel()
    return int(fp), int(fn)
def validation_costs(missed_cost, false_alarm_cost=5):
    records = []
    for threshold in thresholds:
        fp, fn = errors(y_val, p_val >= threshold)
        records.append([threshold, fp, fn, false_alarm_cost*fp + missed_cost*fn])
    return pd.DataFrame(records, columns=['threshold','false_alarms','misses','cost'])
missed_cost, false_alarm_cost = 50, 5
validation = validation_costs(missed_cost, false_alarm_cost)
selected = float(validation.loc[validation['cost'].idxmin(), 'threshold'])
print(validation.to_string(index=False))
print('Selected using validation only:', round(selected, 2))


### Final evaluation - run after the validation decision is fixed

Do your validation experiments before this cell. Do not choose a different threshold because you dislike this result. Compare error costs with the always-negative baseline as well as the conventional 0.5 threshold; report that the latter comparison is descriptive, not a new selection step.


In [ ]:
p_test = model.predict_proba(X_test)[:, 1]
fp, fn = errors(y_test, p_test >= selected)
default_fp, default_fn = errors(y_test, p_test >= .5)
print({'test_n': len(y_test), 'selected_threshold': round(selected, 2),
       'selected_cost': false_alarm_cost*fp + missed_cost*fn,
       'default_0.5_cost': false_alarm_cost*default_fp + missed_cost*default_fn,
       'always_negative_cost': int(y_test.sum())*missed_cost})


## Day 3 - Backtest several cutoffs before making a claim

Compare last-value and weekly-repeat forecasts across four historical origins. Use only observations available at each origin. Select on mean validation MAE; reserve the final fourteen days for one final evaluation. These are executable backtests, extending the earlier diagram-only activity.

**Your experiment:** before final evaluation, inspect the four origins individually. Does the average conceal a bad window? Explain how changing the planning horizon would change the test design. The code below uses a 14-day horizon throughout.


In [ ]:
rng = np.random.default_rng(41)
t = np.arange(126)
demand = 100 + .12*t + 18*np.sin(2*np.pi*t/7) + rng.normal(0,4,len(t))
development, final = demand[:-14], demand[-14:]
def forecast(history, method, horizon):
    if method == 'last': return np.repeat(history[-1], horizon)
    if method == 'weekly': return np.resize(history[-7:], horizon)
    raise ValueError(method)
records = []
for origin in [56,70,84,98]:
    history = development[:origin]
    actual = development[origin:origin+14]
    for method in ['last','weekly']:
        mae = np.abs(actual - forecast(history,method,14)).mean()
        records.append([origin,method,mae])
backtest = pd.DataFrame(records,columns=['origin','method','mae'])
validation_mae = backtest.groupby('method')['mae'].mean()
chosen_method = validation_mae.idxmin()
print(backtest.pivot(index='origin',columns='method',values='mae').round(2).to_string())
print('Mean validation MAE:', validation_mae.round(2).to_dict())
print('Chosen method:', chosen_method)


### Final forecast check - run after the method is fixed

Changing a method after seeing this result contaminates this final period. Future experiments need a fresh untouched evaluation for a fresh claim.


In [ ]:
final_prediction = forecast(development,chosen_method,len(final))
print('Final 14-day MAE:', round(np.abs(final-final_prediction).mean(),2))


## Day 4 - Which assumptions actually change the action?

Use the same observed 8 defects in 100. Compare three deliberately different priors and two action rules. The event of concern is an underlying defect rate above 10%; the policy trigger is a probability above 20% or 30%. Those two thresholds have different meanings.

**Your experiment:** identify decisions that change with the prior or policy. Ask what evidence justifies the prior. Repeat with 80 defects in 1,000, then explain why a larger sample cannot fix biased sampling.


In [ ]:
def sensitivity(defects=8, inspected=100):
    records=[]
    for name,a,b in [('uniform',1,1),('moderate',2,18),('strong concern',20,80)]:
        pa,pb = a+defects,b+inspected-defects
        tail = beta.sf(.1,pa,pb)
        lo,hi = beta.ppf([.025,.975],pa,pb)
        records.append([name,pa/(pa+pb),lo,hi,tail,tail>.2,tail>.3])
    return pd.DataFrame(records,columns=['prior','mean','low95','high95','P(rate>10%)','act_at_20%','act_at_30%'])
print(sensitivity().round(4).to_string(index=False))


## Day 5 - Detect a problem and decide who acts

Monitoring needs a pre-agreed rule and an owner. These synthetic weekly errors assume the true outcomes have already arrived. A 12-minute MAE limit and a two-consecutive-week trigger are fictional operating choices, not recommended universal values.

**Your experiment:** compare limits of 10, 12 and 15 before discussing the observed alerts. Discuss delayed labels, false alarms and which service conditions the average could hide. Design an escalation and rollback action; crossing a threshold does not identify the cause.


In [ ]:
monitoring = pd.DataFrame({'week':np.arange(1,9),'mae_minutes':[7.8,8.1,7.5,9.2,10.1,13.5,14.1,8.4]})
limit=12
monitoring['breach'] = monitoring['mae_minutes'] > limit
monitoring['two_week_trigger'] = monitoring['breach'] & monitoring['breach'].shift(1,fill_value=False)
print(monitoring.to_string(index=False))
print('Trigger weeks:',monitoring.loc[monitoring['two_week_trigger'],'week'].tolist())


## Finish the experiment, then challenge it

Each pair submits: original expectation; change made; observed result; limitation; action. Another pair must identify one assumption or propose a counterexample. Keep the code with the explanation. This is the deliverable, not a screenshot of a successful cell.
